In [71]:
import numpy as np
import pandas as pd

In [72]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [73]:
df = pd.read_csv("covid_toy.csv")
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [74]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    df.drop(columns = ["has_covid"]), df["has_covid"], test_size=0.2, random_state=42
)

In [75]:
df.isnull().sum() # it counts how many NaN or missing values in our dataset

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [76]:
# for missing values: SimpleImputer
sip = SimpleImputer()
X_train_fev = sip.fit_transform(X_train[['fever']])

X_test_fev = sip.transform(X_test[['fever']])


In [77]:
# for ordinal categories: OrdinalEncoder
oec = OrdinalEncoder(categories = [['Mild','Strong']])
X_train_cgh = oec.fit_transform(X_train[['cough']])

X_test_cgh = oec.transform(X_test[['cough']])


In [78]:
# for nominal categories: OneHotEncoder
ohe = OneHotEncoder(drop='first',sparse_output=False)
X_train_gen_city = ohe.fit_transform(X_train[['gender' , 'city']])

X_test_gen_city = ohe.transform(X_test[['gender' , 'city']])

In [79]:
# Extract Age-feature:-
X_train_age = X_train.drop(columns = ['gender','fever','cough','city']).values

X_test_age = X_test.drop(columns = ['gender','fever','cough','city']).values

In [80]:
# Concatinating:-
X_train_final = np.concatenate((X_train_age,X_train_fev,X_train_cgh,X_train_gen_city),axis=1)

X_test_final = np.concatenate((X_test_age,X_test_fev,X_test_cgh,X_test_gen_city),axis=1)

In [81]:
X_train_final.shape

(80, 7)

In [86]:
X_test_final.shape


(20, 7)

In [83]:
# ColumnTransformer:-
from sklearn.compose import ColumnTransformer

In [91]:
transformer = ColumnTransformer(
    transformers = [
        ('trf1', SimpleImputer(), ['fever']),
        ('trf2', OrdinalEncoder(categories = [['Mild','Strong']]),['cough']),
        ('trf3', OneHotEncoder(drop='first',sparse_output=False),['gender','city'])
    ], remainder='passthrough'
)

In [93]:
transformer.fit_transform(X_train).shape

(80, 7)

In [96]:
transformer.transform(X_test).shape

(20, 7)